# Phase 0 — deterministic walking skeleton

A repeatable eight-day synthetic plumbing path. The notebook uses the global Python 3.14 kernel and imports the extracted standard-library package below; no provider or network call occurs unless its explicit gate is changed.

In [1]:
import sys
from dataclasses import dataclass
from pathlib import Path

# This keeps notebook work on the selected global kernel, not the uv test environment.
repo_root = Path.cwd().resolve()
while not (repo_root / "src").is_dir():
    if repo_root.parent == repo_root:
        raise RuntimeError(
            "Open this notebook from the repository or a descendant directory."
        )
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
from reliability.eval.smoke import DeterministicClient, run

print(f"Python: {sys.version.split()[0]}")
print("Imports: extracted local src package (standard library only)")

Python: 3.14.7
Imports: standard library only


In [ ]:
# This is the extracted production path; later cells retain the transparent walkthrough.
package_evidence, package_answer = run(DeterministicClient())
assert package_evidence.value == 3.0
assert package_answer == "The MAE was 3 units."
print("Extracted package path: PASS")

In [2]:
# Define the typed observations, evidence row, and fixed Phase 0 contract constants.
@dataclass(frozen=True)
class Observation:
    day: int
    value: int


@dataclass(frozen=True)
class EvidenceRow:
    row_id: str
    series_id: str
    model_version: str
    origin_day: int
    horizon: int
    metric: str
    value: float
    unit: str


SERIES_ID, MODEL_VERSION = "synthetic_weekly_01", "seasonal_naive_v0"
ORIGIN_DAY, HORIZON, METRIC = 7, 1, "mae"
print(
    f"Contract: {SERIES_ID} | {MODEL_VERSION} | origin={ORIGIN_DAY} | horizon={HORIZON} | metric={METRIC}"
)

Contract: synthetic_weekly_01 | seasonal_naive_v0 | origin=7 | horizon=1 | metric=mae


In [3]:
# Build and verify the fixed eight-day synthetic fixture, separating training days 1–7 from the day-8 target.
FIXTURE_VALUES = (10, 12, 14, 16, 18, 20, 22, 13)
observations = tuple(
    Observation(i + 1, value) for i, value in enumerate(FIXTURE_VALUES)
)
train, target = observations[:7], observations[7:]
assert tuple(x.value for x in train) == (10, 12, 14, 16, 18, 20, 22)
assert tuple(x.value for x in target) == (13,)
assert train[-1].day == 7 and target[0].day == 8
print("Train (days 1–7):", tuple(x.value for x in train))
print("Target (day 8):", tuple(x.value for x in target))

Train (days 1–7): (10, 12, 14, 16, 18, 20, 22)
Target (day 8): (13,)


In [4]:
def seasonal_naive(
    history: tuple[Observation, ...], *, period: int, horizon: int
) -> int:
    if period != 7:
        raise ValueError("Phase 0 requires period=7")
    if horizon != 1:
        raise ValueError("Phase 0 supports horizon=1 only")
    if len(history) < period:
        raise ValueError("history must contain one full period")
    return history[-period].value


def mae(actual: int, predicted: int) -> float:
    return float(abs(actual - predicted))


prediction = seasonal_naive(train, period=7, horizon=1)
actual = target[0].value
mae_value = mae(actual, prediction)
assert prediction == 10
assert (actual, prediction, mae_value) == (13, 10, 3.0)
print(f"Seasonal-naive forecast (period=7, horizon=1): {prediction}")
print(f"MAE: |{actual} - {prediction}| = {mae_value:g}")

Seasonal-naive forecast (period=7, horizon=1): 10
MAE: |13 - 10| = 3


In [5]:
evidence = EvidenceRow(
    "synthetic_weekly_01__seasonal_naive_v0__origin_7__h1__mae",
    SERIES_ID,
    MODEL_VERSION,
    ORIGIN_DAY,
    HORIZON,
    METRIC,
    mae_value,
    "units",
)
evidence_rows = (evidence,)
print(evidence)

EvidenceRow(row_id='synthetic_weekly_01__seasonal_naive_v0__origin_7__h1__mae', series_id='synthetic_weekly_01', model_version='seasonal_naive_v0', origin_day=7, horizon=1, metric='mae', value=3.0, unit='units')


In [6]:
def find_evidence(
    rows: tuple[EvidenceRow, ...],
    *,
    series_id: str,
    model_version: str,
    origin_day: int,
    horizon: int,
    metric: str,
) -> EvidenceRow:
    matches = [
        row
        for row in rows
        if (row.series_id, row.model_version, row.origin_day, row.horizon, row.metric)
        == (series_id, model_version, origin_day, horizon, metric)
    ]
    if len(matches) != 1:
        raise LookupError(f"expected exactly one evidence row, found {len(matches)}")
    return matches[0]


looked_up = find_evidence(
    evidence_rows,
    series_id=SERIES_ID,
    model_version=MODEL_VERSION,
    origin_day=ORIGIN_DAY,
    horizon=HORIZON,
    metric=METRIC,
)
assert looked_up == evidence
print(
    f"Exact lookup: {looked_up.row_id} -> {looked_up.metric}={looked_up.value:g} {looked_up.unit}"
)

Exact lookup: synthetic_weekly_01__seasonal_naive_v0__origin_7__h1__mae -> mae=3 units


## Deterministic boundary

Question: *What was the MAE for synthetic_weekly_01 at origin day 7, horizon 1?*

The fake client below is inspected for this exact question and evidence row. This proves plumbing and response preservation, not LLM reasoning.

In [7]:
QUESTION = "What was the MAE for synthetic_weekly_01 at origin day 7, horizon 1?"
EXPECTED_TEXT = "The MAE was 3 units."


@dataclass(frozen=True)
class Protocol:
    arm: str
    question: str


class ArmA:
    def __init__(self, client: object) -> None:
        self.client = client

    def answer(self, question: str, evidence_row: EvidenceRow) -> str:
        return self.client.answer(question, evidence_row)


class InspectingFakeClient:
    def answer(self, question: str, evidence_row: EvidenceRow) -> str:
        assert question == QUESTION
        assert evidence_row == evidence
        return EXPECTED_TEXT


protocol = Protocol("ArmA", QUESTION)
answer_text = ArmA(InspectingFakeClient()).answer(protocol.question, looked_up)
assert answer_text == EXPECTED_TEXT
print(f"{protocol.arm} E2E response: {answer_text}")

ArmA E2E response: The MAE was 3 units.


In [8]:
class WrongAnswerFakeClient:
    def answer(self, question: str, evidence_row: EvidenceRow) -> str:
        assert question == QUESTION
        assert evidence_row == evidence
        return "99"


wrong_answer_text = ArmA(WrongAnswerFakeClient()).answer(protocol.question, looked_up)
assert wrong_answer_text == "99"
assert wrong_answer_text != EXPECTED_TEXT
print(f"Wrong-answer client text (preserved unchanged): {wrong_answer_text}")

Wrong-answer client text (preserved unchanged): 99


In [9]:
assert (evidence.value, evidence.unit) == (3.0, "units")
assert answer_text == EXPECTED_TEXT
assert wrong_answer_text == "99"
print("PHASE 0 DETERMINISTIC STATUS: PASS")

PHASE 0 DETERMINISTIC STATUS: PASS


## OpenRouter live smoke — recorded one-shot inspection

Provider: OpenRouter (`https://openrouter.ai/api/v1/chat/completions`)  
Model: `deepseek/deepseek-v4-flash-0731`  
Environment variable: `OPENROUTER_API_KEY`

The ignored repo-root `.env` file is never committed, and an earlier one-shot inspection returned `3 units` (1,589 ms). That was a smoke observation only—not an evaluation, reliability result, or forecast-quality claim. `RUN_LIVE_SMOKE` is false by default so a restart or Run All never sends a paid request.

In [10]:
# Deliberately false: notebook Run All must not make a paid network request.
RUN_LIVE_SMOKE = False
if RUN_LIVE_SMOKE:
    from reliability.eval.smoke import main

    main(["--live"])
else:
    print("Live smoke skipped (RUN_LIVE_SMOKE=False).")

LIVE SMOKE INSPECTION — not an evaluation
Selected model: deepseek/deepseek-v4-flash-0731
Raw answer: 3 units
Elapsed: 1589 ms
reported_expected_answer (contains 3 + unit): True
